In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')
print(f'Pandas: {pd.__version__} | NumPy: {np.__version__}')

## Step 1: Load Data

In [ ]:
# Load data files
prices_df = pd.read_csv('VNdata\\prices_df_handled.csv', index_col=0, parse_dates=True)
volume_df = pd.read_csv('VNdata\\volume_df_handled_2.csv', index_col=0, parse_dates=True)
fu_df = pd.read_csv('VNdata\\fu_df_handled.csv', index_col=0, parse_dates=True)

columns = prices_df.columns
volume_df = volume_df[columns]
volume_df = volume_df.loc['2017-08-10': '2025-12-31']
fu_df = fu_df.loc['2017-08-10': '2025-12-31']
print(f'✓ Data loaded:')
print(f'  Prices: {prices_df.shape}')
print(f'  Volumes: {volume_df.shape}')
print(f'  FU_VN30: {fu_df.shape}')

In [ ]:
prices_df.isna().sum().sum(), volume_df.isna().sum().sum(), fu_df.isna().sum().sum()

## Step 2-3: Universe Selection and Returns Computation

In [ ]:
def top_liquidity_stocks(price_df, volume_df, top_n=300, average_window=60):
    # Calculate liquidity metrics (average volume* price over lookback window)
    price_win = price_df.tail(average_window)
    volume_win = volume_df.tail(average_window)

    liquidity = (price_win * volume_win).mean()
    top_stocks = liquidity.nlargest(min(top_n, len(liquidity))).index

    return price_df[top_stocks], volume_df[top_stocks].astype(float)

# --- Fill missing values with average of 2 before and 2 after ---
def fill_na_with_2before_2after(df: pd.DataFrame, min_neighbors=2, require_all=False):
    s1 = df.shift(1)
    s2 = df.shift(2)
    s_1 = df.shift(-1)
    s_2 = df.shift(-2)

    neighbor_sum = s1.fillna(0) + s2.fillna(0) + s_1.fillna(0) + s_2.fillna(0)
    neighbor_count = (
        s1.notna().astype(int)
        + s2.notna().astype(int)
        + s_1.notna().astype(int)
        + s_2.notna().astype(int)
    )

    neighbor_mean = neighbor_sum / neighbor_count.where(neighbor_count > 0)

    if require_all:
        fill_values = neighbor_mean.where(neighbor_count == 4)
    else:
        fill_values = neighbor_mean.where(neighbor_count >= min_neighbors)

    return df.where(df.notna(), fill_values)

# --- Universe selection based on liquidity ---
def select_universe(prices_df: pd.DataFrame, volume_df: pd.DataFrame, top_n=300, lookback_window=252):
    # filter stocks with price data on the given date
    date = prices_df.index[-1] # use the most recent date in the price data
    list_stocks_notna_today = prices_df.columns[prices_df.loc[date].notna()]

    # handle missing values by forward fill then drop na
    prices_df = prices_df.dropna(axis=1, how='any')
    columns = prices_df.columns
    volume_df = volume_df[columns]
    # fill volume missing values with average of previous and next valid values
    volume_df = volume_df.dropna(axis=1, how='any')
    columns = volume_df.columns
    prices_df = prices_df[columns]
    
    # return price and volume for the top N stocks by liquidity
    return top_liquidity_stocks(prices_df, volume_df, top_n=top_n, average_window=lookback_window)

# --- Returns calculation ---
def compute_returns(price_df: pd.DataFrame, volume_df: pd.DataFrame=None, with_volume=False, volume_average_window=10):
    # basic returns calculation
    returns = price_df.pct_change(fill_method=None)
    returns = returns.drop(returns.index[0]) # drop the first row which is NaN after pct_change

    # volume adjustment
    if with_volume and volume_df is not None:
        volume_average = volume_df.shift(1).rolling(volume_average_window).mean().dropna() # volume average based on previous day
        volume_average = volume_average.reindex(returns.index) 
        delta_volume = volume_average.diff().dropna() 
        returns = returns*(volume_average/delta_volume)
    return returns

## Step 4: Rolling PCA Factor Extraction

In [ ]:
def standardize_returns(returns_df):
    """Standardize returns: (R - mean) / std per stock."""
    mean_i = returns_df.mean(axis=0)
    std_i  = returns_df.std(axis=0, ddof=1).replace(0, np.nan)
    return ((returns_df - mean_i) / std_i).dropna(axis=1)

def compute_empirical_correlation(standardized_returns):
    """Compute correlation matrix from standardized returns."""
    M = standardized_returns.shape[0]
    return (standardized_returns.values.T @ standardized_returns.values) / (M - 1)

def eigen_decomposition_sorted(corr_matrix):
    """Eigendecomposition with eigenvalues sorted descending."""
    eigvals, eigvecs = np.linalg.eigh(corr_matrix)
    idx = np.argsort(eigvals)[::-1]
    return eigvals[idx], eigvecs[:, idx]


def select_n_factors(eigvals, variance_threshold=0.70):
    """Chọn k nhỏ nhất sao cho cumulative variance >= threshold."""
    cumvar = np.cumsum(eigvals) / eigvals.sum()
    k      = int(np.searchsorted(cumvar, variance_threshold) + 1)
    k      = max(1, min(k, len(eigvals)))
    return k

def window_pca_engine(returns_df, variance_threshold=0.70):
    """Perform PCA, số factors tự động theo variance_threshold."""
    Z                  = standardize_returns(returns_df)
    C                  = compute_empirical_correlation(Z)
    eigvals, eigvecs   = eigen_decomposition_sorted(C)
    k                  = select_n_factors(eigvals, variance_threshold)
    V                  = eigvecs[:, :k]
    std_stocks         = returns_df.std(ddof=1).replace(0, np.nan)
    Q                  = V / std_stocks.values.reshape(-1, 1)
    variance_explained = float(np.cumsum(eigvals)[k - 1] / eigvals.sum())
    return {
        'Q'                 : Q,
        'V'                 : V,
        'k'                 : k,
        'stocks'            : returns_df.columns,
        'eigenvalues'       : eigvals[:k],
        'variance_explained': variance_explained
    }



## Step 5: Residual Construction

In [ ]:
def compute_pca_residuals(returns_df, pca_dict):
    """Compute PCA residuals via OLS regression on factor returns."""
    Q      = pca_dict['Q']
    stocks = pca_dict['stocks']
    R      = returns_df[stocks].values              # [60 × N_stocks]
    F      = R @ Q                                  # [60 × k] factor returns
    X      = np.column_stack([np.ones(len(R)), F])  # [60 × (k+1)]
    XtX_inv = np.linalg.pinv(X.T @ X)
    B       = XtX_inv @ (X.T @ R)                  # [k+1 × N_stocks]
    residuals = R - X @ B                           # [60 × N_stocks]
    return pd.DataFrame(residuals, index=returns_df.index, columns=stocks)

# def compute_pca_residuals(returns_df: pd.DataFrame, pca_dict: dict, window=60):
#     # take last 'window' returns for regression
#     returns_win = returns_df.iloc[-window:]

#     Q = pca_dict["Q"]
#     stocks = pca_dict["stocks"]

#     # check stocks match
#     if list(stocks) != list(returns_df.columns):
#         raise ValueError("PCA stocks do not match returns stocks")

#     # matrix returns
#     R = returns_win[stocks].values     # (window × N)

#     # factor returns
#     F = R @ Q                                    # (window × k)

#     # design matrix with intercept
#     X = np.column_stack([np.ones(window), F])     # (window × (k+1))

#     # OLS solution
#     XtX_inv = np.linalg.pinv(X.T @ X)
#     B = XtX_inv @ (X.T @ R)                       # ((k+1) × N)

#     # fitted returns
#     fitted = X @ B                                # (window × N)

#     # residuals
#     residuals = R - fitted

#     # convert to DataFrame
#     residuals_df = pd.DataFrame(
#         residuals,
#         index=returns_win.index,
#         columns=stocks
#     )

#     return residuals_df      



## Step 6: OU/AR(1) Mean Reversion Fitting

In [ ]:

def fit_ou_model_on_window(eps_window, min_obs=40, kappa_min=8.4, return_details=True):
    """
    Fit 1 bộ tham số OU cho mỗi cổ phiếu trên chính DataFrame window hiện tại.

    Parameters
    ----------
    eps_window : DataFrame
        DataFrame [window x n_stocks], mỗi cột là residual của 1 stock.
    min_obs : int
        Số quan sát tối thiểu để fit.
    kappa_min : float
        Chỉ giữ các stock có kappa >= kappa_min.
    return_details : bool
        Nếu True trả về (kappa, mu, sigma_eq), ngược lại trả về (mu, sigma_eq).

    Returns
    -------
    kappa : Series
    mu : Series
    sigma_eq : Series
        Index đều là tên stock.
    """

    all_stocks = eps_window.columns.tolist()

    kappa = pd.Series(index=all_stocks, dtype=float)
    mu = pd.Series(index=all_stocks, dtype=float)
    sigma_eq = pd.Series(index=all_stocks, dtype=float)

    for col in all_stocks:
        eps = pd.to_numeric(eps_window[col], errors='coerce').dropna()

        if len(eps) < min_obs:
            continue

        # Giữ nguyên công thức cũ
        Xcum = eps.cumsum().values
        if len(Xcum) < 3:
            continue

        x_prev = Xcum[:-1]
        x_next = Xcum[1:]

        A = np.column_stack([np.ones(len(x_prev)), x_prev])

        try:
            a, b = np.linalg.lstsq(A, x_next, rcond=None)[0]
        except Exception:
            continue

        resid_ar = x_next - (a + b * x_prev)
        var_xi = np.var(resid_ar, ddof=1)

        if not np.isfinite(b) or var_xi <= 1e-12 or b <= 0 or b >= 1.0:
            continue

        kappa_val = -np.log(b) * 252.0
        mu_val = a / (1.0 - b)
        sigma_eq_val = np.sqrt(var_xi / (1.0 - b**2))

        if not (np.isfinite(kappa_val) and np.isfinite(mu_val) and np.isfinite(sigma_eq_val)):
            continue
        if sigma_eq_val <= 1e-12:
            continue
        if kappa_val < kappa_min:
            continue

        kappa.loc[col] = kappa_val
        mu.loc[col] = mu_val
        sigma_eq.loc[col] = sigma_eq_val

    if return_details:
        return kappa, mu, sigma_eq
    return mu, sigma_eq

## Step 7: S-Score Computation

In [ ]:
def compute_s_score(mu: pd.Series, sigma_eq: pd.Series):
    """Compute s-score: s = (eps - mu) / sigma_eq."""
    sscore = -mu / sigma_eq
    return sscore.replace([np.inf, -np.inf], np.nan)

## Step 8: Signal Generation Helpers

In [ ]:
def generate_signals_based_on_score(s_score: pd.Series, entry_cutoff=-1.5, close_cutoff=-0.5):
    """
    Generate long-only signals 
    """
    signal = pd.Series(0, index=s_score.index)
    
    signal[s_score <= entry_cutoff] = 1
    signal[s_score >= close_cutoff] = -1
    signal[(s_score > entry_cutoff) & (s_score < close_cutoff)] = 0

    return signal

In [ ]:
def generate_trading_signals_for_date(prices_df, volume_df, return_df,
                                       top_n=300, lookback_window=252, fit_window=60, 
                                       variance_threshold=0.70, kappa_min=8.4):
    # 1. Universe selection
    price_univ, volume_univ = select_universe(prices_df, volume_df, top_n=top_n, lookback_window=lookback_window)

    # 2. Returns calculation
    returns_univ = return_df[price_univ.columns].loc[price_univ.index]  # align returns with price universe

    # 3. PCA extraction
    pca_dict = window_pca_engine(returns_univ, variance_threshold=variance_threshold)

    # 4. Compute residuals
    residuals_df = compute_pca_residuals(returns_univ, pca_dict)

    # 5. Fit OU model and compute s-score
    kappa, mu, sigma_eq = fit_ou_model_on_window(residuals_df, kappa_min=kappa_min)
    s_score = compute_s_score(mu, sigma_eq)

    # 6. Generate signals
    signals = generate_signals_based_on_score(s_score)

    return signals, price_univ, returns_univ, residuals_df

## Step 9: Hedging by FU VN30

In [ ]:
def exposure_of_portfolio_with_FU(portfolio_allocation, return_price, return_fu):
    '''Compute beta exposure of the portfolio to FU returns.
    portfolio_allocation: Series with stock names as index and allocation amounts as values.
    return_price: DataFrame of returns of the stocks in window used for beta estimation
    return_fu: Series of FU returns.'''
    total_allocation = portfolio_allocation.sum()
    if total_allocation == 0:
        return 0.0
    return_portfolio = (return_price.dot(portfolio_allocation) / total_allocation)
    # beta = cov(portfolio_returns, return_fu) / var(return_fu)
    cov_pf_fu = np.cov(return_portfolio, return_fu)[0, 1]
    var_fu = np.var(return_fu, ddof=1)
    beta = cov_pf_fu / var_fu if var_fu > 0 else np.nan
    return beta

In [ ]:
def N0_of_FU_for_hedge(portfolio_allocation, return_price, return_fu, current_price_fu, M=100):
    beta = exposure_of_portfolio_with_FU(portfolio_allocation, return_price, return_fu)
    if np.isnan(beta) or beta == 0:
        return np.nan
    N0 = -beta * portfolio_allocation.sum() / (current_price_fu*M)
    return N0

In [ ]:
def compute_returns_portfolio(cur_port, cur_return):
    stock_na = (cur_return.isna()).index
    temp_port = cur_port.copy()
    cur_port[stock_na] = 0
    cur_return[stock_na] = 0
    sell_quantity = temp_port.sum() - cur_port.sum()
    port_return = (cur_port * cur_return).sum()
    return port_return, sell_quantity, cur_port

## Step 10: Vietnam Long-Only Backtest (Position-Based Capital Allocation)

In [ ]:
# Kiểm tra fu_df có cột nào
fu_col = [col for col in fu_df.columns if 'price' in col.lower() or 'close' in col.lower()]
fu_col = fu_col[0] if fu_col else fu_df.columns[0]
print(f'Dùng cột futures: {fu_col}')

In [ ]:
def backtest_strategy(price_df, volume_df, fu_df,
                      start_date, end_date,
                      PCA_window=252, PCA_variance_threshold=0.70,
                      fit_model_window=60,
                      OU_kappa_min=8.4,
                      entry_cutoff=-1.5, close_cutoff=-0.5,
                      initial_capital=100000, proportion=0.3,
                      cost_transaction=0.0035, multiple_cost_hedge=2.7, M=100,
                      beta_estimation_window=60):
    
    """Backtest chiến lược trading dựa trên PCA residuals và OU s-score."""
    # 1. Chuẩn bị dữ liệu
    date_range = price_df.index[(price_df.index >= start_date) & (price_df.index <= end_date)]
    price_df = price_df.loc[(start_date - pd.Timedelta(days=PCA_window )):end_date]
    volume_df = volume_df.loc[(start_date - pd.Timedelta(days=PCA_window )):end_date]
    return_df = compute_returns(price_df, with_volume=False) 
    fu_df = fu_df.loc[(start_date - pd.Timedelta(days=PCA_window )):end_date]
    return_fu = compute_returns(fu_df, with_volume=False).dropna().squeeze()
    stocks = price_df.columns

    # initialize 
    portfolio_allocations = pd.DataFrame(0, index=date_range, columns=stocks)
    cash = pd.Series(0, index=date_range)
    cash.iloc[0] = initial_capital
    equity_curve = pd.Series(index=date_range, dtype=float)
    equity_curve.iloc[0] = initial_capital
    actions = pd.DataFrame(0, index=date_range, columns=stocks) # 1: mở long, -1: đóng long, 0: giữ nguyên
    N0_of_FU = pd.Series(0, index=date_range) # số lượng hợp đồng FU để hedge

    # 2. Backtest qua từng ngày
    for t, date in enumerate(date_range):
        print(f'Processing {date.date()} ({t+1}/{len(date_range)})...')

        # 2.1 Chọn universe
        price_window = price_df.loc[:date].tail(PCA_window)
        volume_window = volume_df.loc[:date].tail(PCA_window)
        return_window = return_df.loc[:date].tail(PCA_window)
        fu_window = fu_df.loc[:date].tail(PCA_window)

        # 2.2 Update equity:
        if t > 0:
            returns_day = return_df.loc[date] # returns ngày hiện tại
            return_fu_day = (fu_window.loc[date] - fu_window.loc[date - pd.Timedelta(days=1)])* N0_of_FU.loc[date]*M

            returns_portfolio, cancel_stock_sell_quantity, portfolio_allocations.loc[date] = compute_returns_portfolio(portfolio_allocations.loc[date], returns_day)
            equity_curve.loc[date] = equity_curve.loc[date] + returns_portfolio + return_fu_day + cancel_stock_sell_quantity * cost_transaction 
            cash.loc[date] = cash.loc[date] + return_fu_day + cancel_stock_sell_quantity * (1 - cost_transaction)
            
        else:
            equity_curve.loc[date] = initial_capital
        if date == date_range[-1]:
            print('Reached last date in backtest range.')
            break

        # 2.3: Trading
        signals, price_univ, returns_univ, residuals_df = generate_trading_signals_for_date(price_window, volume_window, return_window,
                                                             top_n=300, lookback_window=PCA_window, fit_window=fit_model_window,
                                                             variance_threshold=PCA_variance_threshold, kappa_min=OU_kappa_min)

        # generate actions based on signals and current portfolio
        for stock in price_univ.columns:    
            signal = signals.loc[stock]
            if (signal == 1) & (portfolio_allocations.loc[date, stock] == 0): # mở long
                actions.loc[date, stock] = 1
            elif (signal==-1) & (portfolio_allocations.loc[date, stock]!=0) & (actions.loc[date - pd.Timedelta(days=1), stock]==0) & (cash.loc[date] > 0): # đóng long
                actions.loc[date, stock] = -1
        
        # quantity for long position:
        if (actions.loc[date] == 1).sum() > 0:
            max_sum_for_trading = cash.loc[date]* proportion
            quantity_for_long = (max_sum_for_trading / (actions.loc[date] == 1).sum())
            long_positions = (actions.loc[date]*quantity_for_long).clip(lower=0)
        portfolio_allocations.loc[date + pd.Timedelta(days=1)] = portfolio_allocations.loc[date] + long_positions
        cost_long = abs(portfolio_allocations.loc[date + pd.Timedelta(days=1)].sum() - portfolio_allocations.loc[date].sum()) * cost_transaction
        equity_curve.loc[date + pd.Timedelta(days=1)] = equity_curve.loc[date] - cost_long
        cash.loc[date + pd.Timedelta(days=1)] = cash.loc[date] - cost_long
        port_before_sell = portfolio_allocations.loc[date + pd.Timedelta(days=1)].sum()

        # update cash and portfolio for closing positions:
        close_stocks = actions.loc[date][actions.loc[date] == -1].index
        for stock in close_stocks:
            cash.loc[date+pd.Timedelta(days=1)] = cash.loc[date] + portfolio_allocations.loc[date, stock]
            portfolio_allocations.loc[date + pd.Timedelta(days=1), stock] = 0
        cost_sell = abs(portfolio_allocations.loc[date + pd.Timedelta(days=1)].sum() - port_before_sell) * cost_transaction
        equity_curve.loc[date + pd.Timedelta(days=1)] -= cost_sell
        cash.loc[date + pd.Timedelta(days=1)] -= cost_sell

        # hedge with FU:
        fu_window_for_beta = return_fu.loc[:date].tail(beta_estimation_window)
        return_window_for_beta = return_window.loc[:date].tail(beta_estimation_window)
        cur_FU = fu_df.loc[date]
        N0_of_FU.loc[date+pd.Timedelta(days=1)] = N0_of_FU_for_hedge(portfolio_allocations.loc[date + pd.Timedelta(days=1)], return_window_for_beta, fu_window_for_beta, cur_FU)
        # update equity and cash for hedge cost:
        cost_hedge_FU = abs(N0_of_FU.loc[date+pd.Timedelta(days=1)] - N0_of_FU.loc[date]) * multiple_cost_hedge
        equity_curve.loc[date + pd.Timedelta(days=1)] -= cost_hedge_FU
        cash.loc[date + pd.Timedelta(days=1)] -= cost_hedge_FU
